In [1]:
%pip install pdfplumber

  Using cached pdfplumber-0.11.9-py3-none-any.whl.metadata (43 kB)
  Using cached pdfminer_six-20251230-py3-none-any.whl.metadata (4.3 kB)
  Using cached pypdfium2-5.6.0-py3-none-win_amd64.whl.metadata (68 kB)
  Using cached charset_normalizer-3.4.5-cp313-cp313-win_amd64.whl.metadata (39 kB)
  Using cached cryptography-46.0.5-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached pdfplumber-0.11.9-py3-none-any.whl (60 kB)
Using cached pdfminer_six-20251230-py3-none-any.whl (6.6 MB)
Using cached charset_normalizer-3.4.5-cp313-cp313-win_amd64.whl (142 kB)
Using cached cryptography-46.0.5-cp311-abi3-win_amd64.whl (3.5 MB)
Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl (183 kB)
Using cached pypdfium2-5.6.0-py3-none-win_amd64.whl (3.7 MB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)

   ---------------------------------------- 0/7 [pypdfium2]
   --

In [ ]:
import os
import pdfplumber
from pathlib import Path

DATA_DIR = Path("data")

def load_documents():
    docs = []
    for path in DATA_DIR.glob("*"):
        if path.suffix == ".txt":
            with open(path, "r", encoding="utf-8") as f:
                docs.append({"filename": path.name, "text": f.read()})

        elif path.suffix == ".pdf":
            text = ""
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    text += page.extract_text() + "\n"
            docs.append({"filename": path.name, "text": text})

    return docs

docs = load_documents()
len(docs), docs[0]["filename"]


(1, 'sharelink.txt')

In [2]:
def chunk_text(text, max_chars=800):
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]
        chunks.append(chunk)
        start = end
    return chunks

chunks = []
for doc in docs:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "source": doc["filename"],
            "text": chunk
        })

len(chunks), chunks[0]

(15,
 {'source': 'sharelink.txt',
  'text': 'Introduction:\nAbout the ShareLink Pro 1000 The ShareLink Pro 1000 Wireless and Wired Collaboration Gateway enables anyone to present wireless or wired content from their computers, tablets, or smartphones onto a display for easy collaboration. It features streaming technology that supports simultaneous display of up to four content sources, including an HDMI-connected device. The HDMI input supports wired connections from any connected source in the room. To support a wide range of environments, the  ShareLink Pro 1000 has collaboration and moderator modes that facilitate both open and restrictive environments. The ShareLink Pro 1000 provides easy integration of AV and mobile devices into meeting, huddle, collaboration, and presentation spaces.\nFeatures:\nWirelessly share content from mobil'})

In [ ]:
%pip install sentence_transformers

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast model

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = model.encode(chunk_texts, convert_to_numpy=True)
chunk_embeddings.shape

d:\MachineLearningCourse\Projects\extron_qa_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2672.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(15, 384)

In [5]:
def retrieve_relevant_chunks(query, top_k=5):
    query_emb = model.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(query_emb, chunk_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in top_idx:
        results.append({
            "score": float(sims[idx]),
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"]
        })
    return results

results = retrieve_relevant_chunks("What is the sharelink used for?")
results[0]

{'score': 0.5181057453155518,
 'source': 'sharelink.txt',
 'text': 'ce. • WebView technology displays slide images on attendee’s personal devices via a Web browser — The ShareLink Pro 1000 enables meeting content to display on a participant’s mobile device. This is ideal for attendees who cannot easily view the main display. • 128-bit data encryption — A variety of security protocols ensure that all content transmitted between devices and the ShareLink Pro 1000 is fully encrypted and secure. • Power over Ethernet (PoE+) allows the ShareLink Pro to receive power and communication over a single Ethernet cable, eliminating the need for a local power supply  Dual Gigabit Ethernet — Provides two high-speed data links, enabling segmentation of guest and private networks for fast and easy access to the web or other network resources. • Fully customizable welcome '}

In [6]:
def answer_question(query, top_k=3):
    results = retrieve_relevant_chunks(query, top_k=top_k)
    print(f"Question: {query}\n")
    print("Most relevant information:\n")
    for r in results:
        print(f"Source: {r['source']} (score: {r['score']:.3f})")
        print(r["text"])
        print("-" * 80)

answer_question("What are the main features of the sharelink?")

Question: What are the main features of the sharelink?

Most relevant information:

Source: sharelink.txt (score: 0.493)
ce. • WebView technology displays slide images on attendee’s personal devices via a Web browser — The ShareLink Pro 1000 enables meeting content to display on a participant’s mobile device. This is ideal for attendees who cannot easily view the main display. • 128-bit data encryption — A variety of security protocols ensure that all content transmitted between devices and the ShareLink Pro 1000 is fully encrypted and secure. • Power over Ethernet (PoE+) allows the ShareLink Pro to receive power and communication over a single Ethernet cable, eliminating the need for a local power supply  Dual Gigabit Ethernet — Provides two high-speed data links, enabling segmentation of guest and private networks for fast and easy access to the web or other network resources. • Fully customizable welcome 
------------------------------------------------------------------------------

In [7]:
def build_context(results):
    context = ""
    for r in results:
        context += f"From {r['source']}:\n{r['text']}\n\n"
    return context

def build_prompt(query, results):
    context = build_context(results)
    prompt = f"""You are an assistant answering questions about Extron products.

Use ONLY the information in the context below. If the answer is not there, say you don't know.

Context:
{context}

Question: {query}
Answer:"""
    return prompt

### Fully Local Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(llm_name)
model_llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    load_in_4bit=True,
    device_map="auto"
)
# model_llm = AutoModelForCausalLM.from_pretrained(llm_name, torch_dtype=torch.float32)

d:\MachineLearningCourse\Projects\extron_qa_proj\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shail\.cache\huggingface\hub\models--microsoft--Phi-3-mini-4k-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█████████

In [34]:
def build_prompt(query, retrieved_chunks):
    context = ""
    for r in retrieved_chunks:
        context += f"Source: {r['source']}\n{r['text']}\n\n"

    prompt = f"""
You are a helpful assistant answering questions about Extron products.

Use ONLY the information in the context below. If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:
"""
    return prompt

In [ ]:
import torch

def generate_answer(prompt, max_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer[len(prompt):].strip()

In [ ]:
def ask_extron(query, top_k=2):
    retrieved = retrieve_relevant_chunks(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)
    answer = generate_answer(prompt)
    return answer
    
    # print("QUESTION:")
    # print(query)
    # print("\nANSWER:")
    # print(answer)
    
    # print("\n--- Retrieved Chunks Used ---")
    # for r in retrieved:
    #     print(f"{r['source']} (score {r['score']:.3f})")

In [26]:
ask_extron("What are the main features of the sharelink?")

'The ShareLink Pro 1000 is a wireless network that allows users to share content from their computers, tablets, or smartphones onto a display for easy collaboration. It includes a wide range of environments, the  ShareLink Pro 1000 is a wireless network that allows users to share content from their computers, tablets, or smartphones onto a display for easy collaboration. The ShareLink Pro 1000 is a wireless network that allows users to share content from their computers, tablets, or smartphones onto a display for easy collaboration. The ShareLink Pro 1000 is a wireless network that allows users to share content from their computers, tablets, or smartphones onto a display for easy collaboration. The ShareLink Pro 1000 is a wireless network that allows users to share content from their computers, tablets, or smartphones onto a display for easy collaboration. The ShareLink Pro 1000 is a wireless network that allows users to share content from their computers, tablets, or smartphones onto 

### Chat History

In [13]:
chat_history = []

In [28]:
def format_chat_history(history):
    text = ""
    for turn in history:
        text += f"User: {turn['user']}\n"
        text += f"Assistant: {turn['assistant']}\n\n"
    return text

In [29]:
def build_chat_prompt(query, retrieved_chunks, history):
    context = ""
    for r in retrieved_chunks:
        context += f"Source: {r['source']}\n{r['text']}\n\n"

    history_text = format_chat_history(history)

    prompt = f"""
You are a helpful assistant answering questions about Extron products.

Use ONLY the information in the context below. If the answer is not in the context, say you don't know.

Conversation so far:
{history_text}

Context:
{context}

User: {query}
Assistant:
"""
    return prompt

In [30]:
import torch

def generate_answer(prompt, max_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    # Decode only the newly generated text
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_output[len(prompt):].strip()
    return answer

In [31]:
def chat(query, top_k=3):
    # Retrieve relevant chunks
    retrieved = retrieve_relevant_chunks(query, top_k=top_k)

    # Build prompt with history
    prompt = build_chat_prompt(query, retrieved, chat_history)

    # Generate answer
    answer = generate_answer(prompt)

    # Save to history
    chat_history.append({
        "user": query,
        "assistant": answer
    })

    # Display
    print(f"User: {query}\n")
    print(f"Assistant: {answer}\n")
    print("--- Retrieved Chunks Used ---")
    for r in retrieved:
        print(f"{r['source']} (score {r['score']:.3f})")

In [18]:
chat("What does the sharelink do?")

User: What does the sharelink do?

Assistant: Source: sharelink.txt
screen — Multiple configuration options to show, hide, or customize information on the welcome screen, so users can quickly connect and begin sharing their content. • Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple video and video sources — Supports multiple

In [19]:
%pip install gradio

  Using cached gradio-6.9.0-py3-none-any.whl.metadata (16 kB)
  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached audioop_lts-0.2.2-cp313-abi3-win_amd64.whl.metadata (2.0 kB)
  Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
  Using cached fastapi-0.135.1-py3-none-any.whl.metadata (30 kB)
  Using cached ffmpy-1.0.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached gradio_client-2.3.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached orjson-3.11.7-cp313-cp313-win_amd64.whl.metadata (43 kB)
  Using cached pandas-3.0.1-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached pillow-12.1.1-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.22-py3-none-any.whl.metadata (1.8 kB)
  Using cached pytz-2026.1.post1-py2.py3-no

In [ ]:
import gradio as gr

def chat_fn(query):
    answer = ask_extron(query)
    return answer

ui = gr.Interface(
    fn=chat_fn,
    inputs=gr.Textbox(label="Ask about Extron products"),
    outputs=gr.Textbox(label="Answer"),
    title="Extron QA Assistant",
    description="Ask any question about Extron products. Powered by local RAG + LLM.",
)

ui.launch()
